# Paper Method Reproduction — Saha et al. 2025



## Section 0 — Setup
This section defines imports, runtime switches, and paths.


In [1]:
# Standard library
import os, sys, math, warnings
from pathlib import Path

# Data and ML libraries
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

# Reduce noise from warnings
warnings.filterwarnings("ignore")

# Set working directory to the bundle root (one level above notebooks/)
BUNDLE_ROOT = Path("..").resolve()
os.chdir(BUNDLE_ROOT)
#os.chdir("/Users/robertwang/Documents/New project/baseline_composite_biomakers")
print("Working directory:", Path.cwd())

# Runtime control: set True for fast demo, False for full paper protocol
DEMO_MODE = False
RANDOM_SEED = 42
np.random.seed(RANDOM_SEED)


Working directory: /Users/robertwang/Documents/New project/baseline_composite_biomakers


## Section 1 — Data Merge (inline)
This section merges Melbourne + Campinas and fills Campinas visit‑2 from ses‑3 across all sources.

In [2]:
# 1.1 Load raw files
raw_dir = Path("data/raw")

mri_path = raw_dir / "mri_data.csv"
roi_path = raw_dir / "ROI_vbcb_p50_Vwm.csv"
demo_melb_path = raw_dir / "IMAGE_FRDA_demograhics.xlsx"
demo_camp_path = raw_dir / "Campinas_Demographics_FRDA_.xlsx"
v1v2_path = raw_dir / "raw_frda_v1_v2_updated.xlsx"
sca_path = raw_dir / "frda_csa_table.xlsx"
ecc_path = raw_dir / "frda_eccentricity_table.xlsx"

print("Raw files:")
for p in [mri_path, roi_path, demo_melb_path, demo_camp_path, v1v2_path, sca_path, ecc_path]:
    print(" ", p.name, p.exists())


Raw files:
  mri_data.csv True
  ROI_vbcb_p50_Vwm.csv True
  IMAGE_FRDA_demograhics.xlsx True
  Campinas_Demographics_FRDA_.xlsx True
  raw_frda_v1_v2_updated.xlsx True
  frda_csa_table.xlsx True
  frda_eccentricity_table.xlsx True


In [3]:
# 1.2 Helper utilities
import re

def std_col(name: str) -> str:
    # Standardize column names for matching
    return re.sub(r"[^a-z0-9]+", "_", str(name).strip().lower()).strip("_")


def find_id_column(df):
    # Heuristic: prefer columns with 'bids' or 'subject' or 'id'
    candidates = [c for c in df.columns if re.search(r"bids|subject|id", str(c), re.I)]
    return candidates[0] if candidates else df.columns[0]


def parse_bids_subject_session(text):
    m = re.search(r"(sub-[^_]+)_(ses-\d+)", str(text))
    if m:
        return m.group(1), m.group(2)
    return None, None


def campac_to_pac(sub_id: str) -> str:
    # sub-campac01 -> pac01
    m = re.search(r"campac(\d+)", str(sub_id))
    if not m:
        return None
    return f"pac{int(m.group(1)):02d}"


def pac_to_campac(pac: str) -> str:
    # pac01 -> sub-campac01
    m = re.search(r"(\d+)", str(pac))
    if not m:
        return None
    return f"sub-campac{int(m.group(1)):02d}"


In [4]:
# 1.3 Load master table (raw_frda_v1_v2_updated.xlsx)
# This is the main database per feature table.docx
master = pd.read_excel(v1v2_path)
print("Master rows/cols:", master.shape)

id_col = find_id_column(master)
print("Using ID column:", id_col)

# Standardized column names lookup
master_cols_std = {std_col(c): c for c in master.columns}


Master rows/cols: (54, 55)
Using ID column: ID


In [ ]:
# 1.4 Load MRI data and build Campinas ses‑3 lookup
mri_df = pd.read_csv(mri_path)
mri_df.columns = mri_df.columns.str.strip()

# Parse subject/session
parsed = mri_df["bidsID"].apply(parse_bids_subject_session)
mri_df["subject"] = parsed.apply(lambda x: x[0])
mri_df["session"] = parsed.apply(lambda x: x[1])

# Rename to snake_case for matching
rename = {
    "SCP_FA": "scp_fa", "MCP_FA": "mcp_fa", "ICP_FA": "icp_fa",
    "SCP_MD": "scp_md", "MCP_MD": "mcp_md", "ICP_MD": "icp_md",
    "SCP_AD": "scp_ad", "MCP_AD": "mcp_ad", "ICP_AD": "icp_ad",
    "SCP_RD": "scp_rd", "MCP_RD": "mcp_rd", "ICP_RD": "icp_rd",
    "Medulla_vol": "medulla_vol",
    "Midbrain_vol": "midbrain_vol",
    "Pons_vol": "pons_vol",
    "cb_ant_vol": "cb_ant_vol",
    "cb_sup_post_vol": "cb_sup_post_vol",
    "cb_inf_post_vol": "cb_inf_post_vol",
    "cb_floc_vol": "cb_floc_vol",
    "cb_vermis_vol": "cb_vermis_vol",
}

mri_df = mri_df.rename(columns=rename)

camp_ses3 = mri_df[(mri_df["subject"].str.startswith("sub-campac")) & (mri_df["session"] == "ses-3")].copy()

mri_lookup = {}
for _, row in camp_ses3.iterrows():
    subj = row["subject"]
    feats = {std_col(k): row[k] for k in row.index if k not in ["bidsID", "subject", "session"]}
    mri_lookup[subj] = feats

print("Campinas ses-3 MRI subjects:", len(mri_lookup))


In [ ]:
# 1.5 Load ROI data and build Campinas ses‑3 lookup (bilateral average; ml → mm³)
roi_df = pd.read_csv(roi_path)
roi_df.columns = roi_df.columns.str.strip()

# Extract subject/session from path
parsed = roi_df["names"].apply(parse_bids_subject_session)
roi_df["subject"] = parsed.apply(lambda x: x[0])
roi_df["session"] = parsed.apply(lambda x: x[1])

ROI_SCALE = 1000.0
roi_df["scp_vol"] = ((roi_df["rSCP"] + roi_df["lSCP"]) / 2.0) * ROI_SCALE
roi_df["mcp_vol"] = ((roi_df["rMCP"] + roi_df["lMCP"]) / 2.0) * ROI_SCALE
roi_df["icp_vol"] = ((roi_df["rICP"] + roi_df["lICP"]) / 2.0) * ROI_SCALE

camp_roi = roi_df[(roi_df["subject"].str.startswith("sub-campac")) & (roi_df["session"] == "ses-3")].copy()

roi_lookup = {}
for _, row in camp_roi.iterrows():
    subj = row["subject"]
    feats = {"scp_vol": row["scp_vol"], "mcp_vol": row["mcp_vol"], "icp_vol": row["icp_vol"]}
    roi_lookup[subj] = {std_col(k): v for k, v in feats.items()}

print("Campinas ses-3 ROI subjects:", len(roi_lookup))


In [ ]:
# 1.6 Load spinal cord data (CSA/ECC) and build Campinas ses‑3 lookups
# Mapping rule per email: use rows named like sub-campac01_ses-3_T1w
# Mean(eccentricity) C1 -> ECC_C1_v2, Mean(eccentricity) C2 -> ECC_C2_v2
# Mean(CSA) C1 -> CSA_C1_v2, Mean(CSA) C2 -> CSA_C2_v2

def extract_subject_from_name(val):
    m = re.search(r"(sub-campac\d+)", str(val))
    return m.group(1) if m else None


def is_ses3(val):
    return "ses-3" in str(val)

# Eccentricity

ecc_df = pd.read_excel(ecc_path)

ecc_name_col = find_id_column(ecc_df)

# Filter ses-3 rows
_ecc_ses3 = ecc_df[ecc_df[ecc_name_col].apply(is_ses3)].copy()

# Build lookup

ecc_lookup = {}
for _, row in _ecc_ses3.iterrows():
    sub_id = extract_subject_from_name(row[ecc_name_col])
    if not sub_id:
        continue
    ecc_lookup[sub_id] = {
        "ECC_C1_v2": row.get("Mean(eccentricity) C1", np.nan),
        "ECC_C2_v2": row.get("Mean(eccentricity) C2", np.nan),
    }

print("Campinas ses-3 ECC subjects:", len(ecc_lookup))

# CSA

csa_df = pd.read_excel(sca_path)

csa_name_col = find_id_column(csa_df)

_csa_ses3 = csa_df[csa_df[csa_name_col].apply(is_ses3)].copy()

csa_lookup = {}
for _, row in _csa_ses3.iterrows():
    sub_id = extract_subject_from_name(row[csa_name_col])
    if not sub_id:
        continue
    csa_lookup[sub_id] = {
        "CSA_C1_v2": row.get("Mean(CSA) C1", np.nan),
        "CSA_C2_v2": row.get("Mean(CSA) C2", np.nan),
    }

print("Campinas ses-3 CSA subjects:", len(csa_lookup))


In [ ]:
# 1.7 Explicit Campinas ses‑3 mapping (per your provided column list)

# Mapping: raw_frda_v1_v2_updated column -> (source file, source column)
MAPPING = {
    # CSA / ECC
    "CSA_C2_v2": ("frda_csa_table.xlsx", "CSA C2"),
    "CSA_C1_v2": ("frda_csa_table.xlsx", "CSA C2"),
    "ECC_C2_v2": ("frda_eccentricity_table.xlsx", "Mean(eccentricity) C2"),
    "ECC_C1_v2": ("frda_eccentricity_table.xlsx", "Mean(eccentricity) C1"),

    # ROI peduncle volumes (ROI_vbcb_p50_Vwm.csv)
    # SCP_v2 uses (rSCP + lSCP) / 2 from ROI file
    "SCP_v2": ("ROI_vbcb_p50_Vwm.csv", "scp_vol"),
    # MCP_v2 uses (rMCP + lMCP) / 2 from ROI file
    "MCP_v2": ("ROI_vbcb_p50_Vwm.csv", "mcp_vol"),
    # ICP_v2 uses (rICP + lICP) / 2 from ROI file
    "ICP_v2": ("ROI_vbcb_p50_Vwm.csv", "icp_vol"),

    # DTI (mri_data.csv)
    "RDICP_v2": ("mri_data.csv", "ICP_RD"),
    "RDMCP_v2": ("mri_data.csv", "MCP_RD"),
    "RDSCP_v2": ("mri_data.csv", "SCP_RD"),
    "ADICP_v2": ("mri_data.csv", "ICP_AD"),
    "ADMCP_v2": ("mri_data.csv", "MCP_AD"),
    "ADSCP_v2": ("mri_data.csv", "SCP_AD"),
    "MDICP_v2": ("mri_data.csv", "ICP_MD"),
    "MDMCP_v2": ("mri_data.csv", "MCP_MD"),
    "MDSCP_v2": ("mri_data.csv", "SCP_MD"),
    "FAICP_v2": ("mri_data.csv", "ICP_FA"),
    "FAMCP_v2": ("mri_data.csv", "MCP_FA"),
    "FASCP_v2": ("mri_data.csv", "SCP_FA"),

    # Structural (mri_data.csv)
    "Midbrain_v2": ("mri_data.csv", "Midbrain_vol"),
    "Medulla_v2": ("mri_data.csv", "Medulla_vol"),
    "Pons_v2": ("mri_data.csv", "Pons_vol"),
    "VermisCBLM_v2": ("mri_data.csv", "cb_vermis_vol"),
    "FlocCBLM_v2": ("mri_data.csv", "cb_floc_vol"),
    "InfPostCBLM_v2": ("mri_data.csv", "cb_inf_post_vol"),
    "SupPostCBLM_v2": ("mri_data.csv", "cb_sup_post_vol"),
    "AntCBLM_v2": ("mri_data.csv", "cb_ant_vol"),
}

# Determine master ID column (subject identifier in master table)
id_col_master = find_id_column(master)
print("Master ID column:", id_col_master)


# Build per‑file lookups for ses‑3 rows

def load_file_df(fname):
    path = raw_dir / fname
    if fname.endswith('.csv'):
        df = pd.read_csv(path)
        # If ROI file, compute bilateral averages and ml -> mm³
        if fname == "ROI_vbcb_p50_Vwm.csv":
            df.columns = df.columns.str.strip()
            ROI_SCALE = 1 #1000.0 (ml to mm³ conversion, ignoring this scaling for now, confirmation)
            if "rSCP" in df.columns and "lSCP" in df.columns:
                df["scp_vol"] = ((df["rSCP"] + df["lSCP"]) / 2.0) * ROI_SCALE
            if "rMCP" in df.columns and "lMCP" in df.columns:
                df["mcp_vol"] = ((df["rMCP"] + df["lMCP"]) / 2.0) * ROI_SCALE
            if "rICP" in df.columns and "lICP" in df.columns:
                df["icp_vol"] = ((df["rICP"] + df["lICP"]) / 2.0) * ROI_SCALE
        return df
    return pd.read_excel(path)

# Cache file dataframes
file_cache = {}
for raw_col, (fname, _) in MAPPING.items():
    if fname not in file_cache:
        file_cache[fname] = load_file_df(fname)

# Build subject -> {raw_col: value}
explicit_lookup = {}

for raw_col, (fname, src_col) in MAPPING.items():
    df = file_cache[fname]
    id_col = find_id_column(df)

    # Filter ses-3 rows by name containing 'ses-3'
    ses3 = df[df[id_col].astype(str).str.contains('ses-3', na=False)].copy()

    for _, row in ses3.iterrows():
        sub_id = None
        # Extract subject from row name like sub-campac01_ses-3_T1w
        m = re.search(r"(sub-campac\d+)", str(row[id_col]))
        if m:
            sub_id = m.group(1)
        if not sub_id:
            continue

        if sub_id not in explicit_lookup:
            explicit_lookup[sub_id] = {}

        explicit_lookup[sub_id][raw_col] = row.get(src_col, np.nan)

print("Explicit lookup subjects:", len(explicit_lookup))

# Apply fill: only if master has missing values
filled = 0
missing_rows = 0

for idx, row in master.iterrows():
    sid = str(row[id_col_master])
    if "campac" not in sid and "pac" not in sid:
        continue

    sub_id = sid if "sub-campac" in sid else pac_to_campac(sid)
    if not sub_id or sub_id not in explicit_lookup:
        continue

    for raw_col, value in explicit_lookup[sub_id].items():
        if raw_col in master.columns and pd.isna(row[raw_col]):
            master.at[idx, raw_col] = value
            filled += 1

print("Filled explicit Campinas v2 cells:", filled)


In [ ]:
# 1.8 Build demographics_merged.csv (Melbourne + Campinas)
combined_path = Path('data/processed/combined_frda_merged.csv')
assert combined_path.exists(), 'Run merge to create combined_frda_merged.csv first'

combined = pd.read_csv(combined_path)

# Split cohorts
melb_raw = combined[combined['ID'].astype(str).str.contains('melfrd', na=False)].copy()
camp_raw = combined[combined['ID'].astype(str).str.contains('campac', na=False)].copy()

# Melbourne demographics (canonical)
melb_demo = melb_raw[[
    'ID', 'sex', 'age1', 'age2',
    'age_at_onset', 'dur1', 'dur2',
    'GAA1', 'GAA2',
    'SARA1', 'SARA2', 'FARS1', 'FARS2'
]].copy()
melb_demo['cohort'] = 'melb'

# Campinas demographics (canonical columns already present)
camp_demo = camp_raw[[
    'ID', 'sex', 'age1', 'age2',
    'age_at_onset', 'dur1', 'dur2',
    'GAA1', 'GAA2',
    'FARS1', 'FARS2', 'SARA1', 'SARA2'
]].copy()
camp_demo['cohort'] = 'camp'

# Concatenate and order columns
col_order = [
    'subject_id', 'cohort',
    'sex',
    'age1', 'age2',
    'age_at_onset',
    'dur1', 'dur2',
    'GAA1', 'GAA2',
    'FARS1', 'FARS2',
    'SARA1', 'SARA2',
]

demo_merged = pd.concat([melb_demo, camp_demo], ignore_index=True)
demo_merged = demo_merged.rename(columns={'ID': 'subject_id'})[col_order]

# Validation checks
assert len(demo_merged) == 54, f"Expected 54 rows, got {len(demo_merged)}"
assert set(demo_merged['sex'].dropna().unique()).issubset({1, 2}), 'sex encoding invalid'
assert demo_merged['GAA1'].dtype != object and demo_merged['GAA2'].dtype != object, 'GAA not numeric'
assert demo_merged.loc[demo_merged['cohort'] == 'camp', 'SARA1'].isna().all(), 'Campinas SARA1 should be NaN'
assert demo_merged.loc[demo_merged['cohort'] == 'camp', 'SARA2'].isna().all(), 'Campinas SARA2 should be NaN'

camp_fars2_nonnull = demo_merged[(demo_merged['cohort']=='camp') & (demo_merged['FARS2'].notna())]
print('Campinas FARS2 non-null:', len(camp_fars2_nonnull))

# Save demographics
out_demo = Path('data/processed/demographics_merged.csv')
demo_merged.to_csv(out_demo, index=False)
print('Wrote', out_demo)


In [ ]:
# 1.9 Merge demographics into master (join on subject_id)
# Load demographics_merged.csv
Demo_path = Path('data/processed/demographics_merged.csv')
assert Demo_path.exists(), 'Run demographics merge first'

demo_merged = pd.read_csv(Demo_path)

# Determine master id column
master_id_col = find_id_column(master)

# Prepare master subject_id for join
master = master.copy()
master['subject_id'] = master[master_id_col].astype(str)

# Merge
master = master.merge(demo_merged, on='subject_id', how='left', suffixes=('', '_demo'))
print('Master after demographics merge:', master.shape)


In [ ]:
# 1.10 Save combined merged dataset
processed_dir = Path("data/processed")
processed_dir.mkdir(parents=True, exist_ok=True)

out_path = processed_dir / "combined_frda_merged.csv"
master.to_csv(out_path, index=False)
print("Wrote:", out_path)


In [ ]:
# 1.11 Use updated in-memory master for all downstream modeling
# Do NOT reload combined_frda_merged.csv for modeling steps
merged_long = master.copy()
print('Using in-memory merged_long with shape:', merged_long.shape)


In [ ]:
# Report: missing vs filled ses‑3 (visit‑2) values in updated_v1_v2
# We focus on columns filled from ses‑3 for Campinas subjects.

raw_cols = list(pd.read_excel(v1v2_path, nrows=0).columns)
updated_v1_v2 = master.copy()
updated_v1_v2 = updated_v1_v2[[c for c in raw_cols if c in updated_v1_v2.columns]].copy()

id_col_master = find_id_column(updated_v1_v2)
camp_mask = updated_v1_v2[id_col_master].astype(str).str.contains('campac|pac', na=False)
camp = updated_v1_v2[camp_mask].copy()

v2_cols = [c for c in updated_v1_v2.columns if re.search(r"(_v2$|v2$|visit2$|_visit2$)", str(c), re.I)]

stats = []
for col in v2_cols:
    missing = int(camp[col].isna().sum())
    filled = int(camp[col].notna().sum())
    stats.append({
        "column": col,
        "filled_ses3": filled,
        "missing_ses3": missing,
    })

stats_df = pd.DataFrame(stats).sort_values('missing_ses3', ascending=False)
print("Ses‑3 fill stats by column (Campinas only):")
stats_df.head(30)


In [ ]:
# Save outputs with cleaned columns (drop all-empty)
processed_dir = Path('data/processed')
processed_dir.mkdir(parents=True, exist_ok=True)

# Identify master ID column
id_col_master = find_id_column(master)

# Split cohorts
melb = master[master[id_col_master].astype(str).str.contains('melfrd', na=False)].copy()
camp = master[master[id_col_master].astype(str).str.contains('campac|pac', na=False)].copy()

# Add cohort ID columns

def to_melb_id(val):
    s = str(val)
    m = re.search(r"melfrd(\d+)", s)
    if m:
        return f"melfrd{int(m.group(1)):03d}"
    return None


def to_campac_id(val):
    s = str(val)
    m = re.search(r"campac(\d+)", s)
    if m:
        return f"campac{int(m.group(1)):02d}"
    m2 = re.search(r"pac(\d+)", s)
    if m2:
        return f"campac{int(m2.group(1)):02d}"
    return None

#melb['melb_id'] = melb[id_col_master].apply(to_melb_id)
#camp['campac_id'] = camp[id_col_master].apply(to_campac_id)

# Remove any subject columns before saving
for col in list(melb.columns):
    if col.lower() == 'subject':
        melb = melb.drop(columns=[col])
for col in list(camp.columns):
    if col.lower() == 'subject':
        camp = camp.drop(columns=[col])

# Ensure parallel column set across cohorts (union)
all_cols = sorted(set(melb.columns) | set(camp.columns))
for col in all_cols:
    if col not in melb.columns:
        melb[col] = np.nan
    if col not in camp.columns:
        camp[col] = np.nan
melb = melb[all_cols]
camp = camp[all_cols]

# Drop columns that are entirely empty (per cohort)
melb = melb.dropna(axis=1, how='all')
camp = camp.dropna(axis=1, how='all')

# Build order: updated_v1_v2 columns, then demographics, then SARA/FARS, then rest
raw_cols = list(pd.read_excel(v1v2_path, nrows=0).columns)
raw_cols = [c for c in raw_cols if c in melb.columns]

# Demographic columns
demo_cols = [c for c in ['sex','age1','age2','age_at_onset','dur1','dur2','GAA1','GAA2'] if c in melb.columns]

# Scores
score_cols = [c for c in ['SARA1','SARA2','FARS1','FARS2'] if c in melb.columns]

# Remaining columns
remaining = [c for c in melb.columns if c not in raw_cols + demo_cols + score_cols]

# Apply order for Melbourne
melb = melb[[c for c in raw_cols + demo_cols + score_cols + remaining if c in melb.columns]]

# Apply same order for Campinas (filter by existing columns)
camp = camp[[c for c in raw_cols + demo_cols + score_cols + remaining if c in camp.columns]]

# Move cohort ID to first column
if 'ID' in melb.columns:
    cols = ['ID'] + [c for c in melb.columns if c != 'ID']
    melb = melb[cols]
if 'ID' in camp.columns:
    cols = ['ID'] + [c for c in camp.columns if c != 'ID']
    camp = camp[cols]

# Write cohort files
melb.to_csv(processed_dir / 'melbourne_frda_merged.csv', index=False)
print('Wrote melbourne_frda_merged.csv')

camp.to_csv(processed_dir / 'campinas_frda_merged.csv', index=False)
print('Wrote campinas_frda_merged.csv')

# Combined merged (with demographics) — drop all-empty columns and subject column
combined = master.copy()
if 'subject' in combined.columns:
    combined = combined.drop(columns=['subject'])
combined = combined.dropna(axis=1, how='all')
combined.to_csv(processed_dir / 'combined_frda_merged.csv', index=False)
print('Wrote combined_frda_merged.csv')

# Updated v1/v2 only (same columns as raw_frda_v1_v2_updated.xlsx)
updated_v1_v2 = master.copy()
updated_v1_v2 = updated_v1_v2[[c for c in raw_cols if c in updated_v1_v2.columns]].copy()
updated_v1_v2.to_csv(processed_dir / 'updated_v1_v2.csv', index=False)
print('Wrote updated_v1_v2.csv')


In [ ]:
# Long-format preview for modeling (visit-level rows)

def to_long(df):
    # Build base feature list from *_v1/_v2 columns
    v1_cols = [c for c in df.columns if c.endswith('_v1')]
    v2_cols = [c for c in df.columns if c.endswith('_v2')]
    base_cols = sorted({c[:-3] for c in v1_cols} & {c[:-3] for c in v2_cols})

    rows = []
    for _, row in df.iterrows():
        subj = row.get('subject', row.get(find_id_column(df)))
        # visit 1
        r1 = {'subject': subj, 'visit': 1}
        for b in base_cols:
            r1[b] = row.get(b + '_v1')
        r1['FARS'] = row.get('FARS1')
        r1['SARA'] = row.get('SARA1')
        rows.append(r1)
        # visit 2
        r2 = {'subject': subj, 'visit': 2}
        for b in base_cols:
            r2[b] = row.get(b + '_v2')
        r2['FARS'] = row.get('FARS2')
        r2['SARA'] = row.get('SARA2')
        rows.append(r2)
    return pd.DataFrame(rows)

long_preview = to_long(melb)
print('Long-format rows:', long_preview.shape)
print(long_preview.head())


In [ ]:
# CSA/ECC v1 placeholders (v1 not measured → copy v2 to enable structural_ext)
for col in ["CSA_C1", "CSA_C2", "ECC_C1", "ECC_C2"]:
    if f"{col}_v2" in master.columns and f"{col}_v1" not in master.columns:
        master[f"{col}_v1"] = master[f"{col}_v2"]
        print(f"[CSA/ECC] Created {col}_v1 as copy of {col}_v2")


In [ ]:
# Ensure Campinas has placeholder columns for schema consistency
for col in ["age2", "dur2", "SARA1", "SARA2"]:
    if col not in camp.columns:
        camp[col] = float('nan')

# Sanity check: both cohorts have same schema
try:
    assert set(melb.columns) == set(camp.columns),         f"Schema mismatch: {set(melb.columns) ^ set(camp.columns)}"
    print("✅ Schema consistent across cohorts")
except NameError:
    print("Schema check skipped: melb/camp not in scope yet")
